In [51]:
# imports

import pandas as pd
import plotly.express as px
import plotly.io as pio

# open figures in a full browser tab instead of the cramped notebook output
# pane; switch back to "vscode" to render inline again
pio.renderers.default = "browser"

In [52]:
from pathlib import Path

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]

# The personal exports and the public expansion dataset describe the same
# things under different names, so map the expansion columns onto the export
# schema as the file is read
EXPANSION_COLUMNS = {
    "track_id": "Track URI",
    "track_name": "Track Name",
    "album_name": "Album Name",
    "artists": "Artist Name(s)",
    "popularity": "Popularity",
    "duration_ms": "Duration (ms)",
    "explicit": "Explicit",
    "track_genre": "Genres",
    "danceability": "Danceability",
    "energy": "Energy",
    "key": "Key",
    "loudness": "Loudness",
    "mode": "Mode",
    "speechiness": "Speechiness",
    "acousticness": "Acousticness",
    "instrumentalness": "Instrumentalness",
    "liveness": "Liveness",
    "valence": "Valence",
    "tempo": "Tempo",
    "time_signature": "Time Signature",
}

# Present in the exports, absent from the expansion dataset
EXPANSION_MISSING = ["Release Date", "Added By", "Added At", "Record Label"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


def normalize_expansion(df):
    """Reshape the public expansion CSV to match the personal exports."""
    # keep only the columns that map onto the export schema, renamed
    df = df.rename(columns=EXPANSION_COLUMNS)[list(EXPANSION_COLUMNS.values())]

    # bare ids there, full URIs in the exports
    df["Track URI"] = "spotify:track:" + df["Track URI"].astype(str)

    # several artists are joined with ";" there and "," in the exports
    df["Artist Name(s)"] = (
        df["Artist Name(s)"].astype(
            "string").str.replace(";", ",", regex=False)
    )

    # that dataset carries one row per track-genre pair, so a song in several
    # genres repeats; collapse to one row per track with the genres comma
    # joined the way the exports list them
    genres = (
        df.groupby("Track URI")["Genres"]
        .apply(lambda values: ",".join(sorted(set(values.dropna()))))
        .rename("Genres")
    )
    df = (
        df.drop(columns="Genres")
        .drop_duplicates(subset="Track URI")
        .merge(genres, on="Track URI")
    )

    # the export only columns have no counterpart here
    for column in EXPANSION_MISSING:
        df[column] = pd.NA
    return df


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    if "track_id" in df.columns:
        # the public expansion dataset, not one of our own exports
        df = normalize_expansion(df)
        listener = "Expansion"
    else:
        # Listener name is the first word of the file name,
        # e.g. "Logan Exported Data.csv" -> "Logan"
        listener = path.stem.split()[0]
    df["Listener"] = listener
    print(f"{path.name}: {len(df)} rows as {listener}")
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()

14k Dataset Expansion.csv: 89741 rows as Expansion
Amadou Music Taste Export.csv: 1663 rows as Amadou
Logan Exported Data.csv: 1534 rows as Logan
(92938, 25)
Listener
Expansion    89741
Amadou        1663
Logan         1534
Name: count, dtype: int64


,Track URI,Track Name,Album Name,Artist Name(s),Popularity,Duration (ms),Explicit,Danceability,Energy,Key,...,Liveness,Valence,Tempo,Time Signature,Genres,Release Date,Added By,Added At,Record Label,Listener
0,spotify:track:5SuOikwiRyPMVoIQDJUgSV,Comedy,Comedy,Gen Hoshino,73,230666,False,0.676,0.4610,1.0,...,0.3580,0.715,87.917,4.0,"acoustic,j-pop,singer-songwriter,songwriter",<NA>,<NA>,<NA>,<NA>,Expansion
1,spotify:track:4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ghost (Acoustic),Ben Woodward,55,149610,False,0.420,0.1660,1.0,...,0.1010,0.267,77.489,4.0,"acoustic,chill",<NA>,<NA>,<NA>,<NA>,Expansion
2,spotify:track:1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,To Begin Again,"Ingrid Michaelson,ZAYN",57,210826,False,0.438,0.3590,0.0,...,0.1170,0.120,76.332,4.0,acoustic,<NA>,<NA>,<NA>,<NA>,Expansion
3,spotify:track:6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Crazy Rich Asians (Original Motion Picture Sou...,Kina Grannis,71,201933,False,0.266,0.0596,0.0,...,0.1320,0.143,181.740,3.0,acoustic,<NA>,<NA>,<NA>,<NA>,Expansion
4,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Hold On,Chord Overstreet,82,198853,False,0.618,0.4430,2.0,...,0.0829,0.167,119.949,4.0,acoustic,<NA>,<NA>,<NA>,<NA>,Expansion


### Data Cleaning

In [53]:
# check for duplicate Track URIs before anything downstream relies on them

# A listener's export can list the same song more than once, e.g. when it
# sits in several of their playlists, so count the repeats per listener
# dropna=False so rows with no URI at all show up instead of vanishing
print(f"{data['Track URI'].isna().sum()} rows have no Track URI")
uri_counts = data.groupby(["Listener", "Track URI"], dropna=False).size()
repeats = uri_counts[uri_counts > 1]
print(f"{len(repeats)} Track URIs appear more than once for the same listener")
print(f"{repeats.sum() - len(repeats)} extra rows to drop")

# show a few of the worst offenders so we can eyeball what they are
if not repeats.empty:
    worst = repeats.sort_values(ascending=False).head(10).index
    display(
        data[
            [
                pair in set(worst)
                for pair in zip(data["Listener"], data["Track URI"])
            ]
        ].sort_values(["Listener", "Track URI"])[
            ["Listener", "Track URI", "Track Name", "Artist Name(s)"]
        ]
    )

# keep the first copy of each song per listener; the song level columns are
# identical across copies, and the Listener flags in a later cell still need
# one row per track-listener pair
data = data.drop_duplicates(subset=["Listener", "Track URI"])
print(data.shape)

0 rows have no Track URI
0 Track URIs appear more than once for the same listener
0 extra rows to drop
(92938, 25)


In [54]:
# identify missing values in each column
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

Added By            91275
Release Date        89742
Record Label        89741
Added At            89741
Genres                990
Artist Name(s)          2
Danceability            1
Album Name              1
Track Name              1
Valence                 1
Liveness                1
Instrumentalness        1
Acousticness            1
Energy                  1
Key                     1
Mode                    1
Loudness                1
Speechiness             1
Time Signature          1
Tempo                   1
Track URI               0
Duration (ms)           0
Explicit                0
Popularity              0
Listener                0
dtype: int64

In [55]:
# handle the 1 missing values row first

# display the row with missing values - NOT THE GENRES MISSING VALUES
data[data.isnull().any(axis=1) & data['Mode'].isnull()]

,Track URI,Track Name,Album Name,Artist Name(s),Popularity,Duration (ms),Explicit,Danceability,Energy,Key,...,Liveness,Valence,Tempo,Time Signature,Genres,Release Date,Added By,Added At,Record Label,Listener
91184,spotify:episode:3bX59hUSv3Lh3vBxmmmcNP,hatinahh,Un dropped,<NA>,0,95062,True,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:31:43Z,undefined,Amadou


In [56]:
# the episode row above is missing its whole audio-feature block, so drop by null
# count instead of by URI to catch any other rows that are just as incomplete

# Added By and Genres are missing for plenty of good rows, so ignore them here
null_counts = data.drop(columns=['Added By', 'Genres']).isnull().sum(axis=1)
null_drop_threshold = 10
bad_rows = null_counts >= null_drop_threshold

print(f"dropping {bad_rows.sum()} row(s) with {null_drop_threshold}+ nulls")
print(data.loc[bad_rows, ['Track URI', 'Track Name', 'Listener']])
data = data[~bad_rows]

dropping 1 row(s) with 10+ nulls
                                    Track URI Track Name Listener
91184  spotify:episode:3bX59hUSv3Lh3vBxmmmcNP   hatinahh   Amadou


In [57]:
# rerun missing values check

missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

Added By            91275
Added At            89741
Release Date        89741
Record Label        89741
Genres                989
Artist Name(s)          1
Album Name              1
Track Name              1
Track URI               0
Energy                  0
Danceability            0
Explicit                0
Duration (ms)           0
Popularity              0
Key                     0
Mode                    0
Loudness                0
Valence                 0
Liveness                0
Instrumentalness        0
Acousticness            0
Speechiness             0
Time Signature          0
Tempo                   0
Listener                0
dtype: int64

In [58]:
# drop added by column as this is an unneeded column for our analysis
data = data.drop(columns=['Added By'])

In [59]:
# rerun missing values check
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

Release Date        89741
Added At            89741
Record Label        89741
Genres                989
Track Name              1
Album Name              1
Artist Name(s)          1
Track URI               0
Danceability            0
Explicit                0
Duration (ms)           0
Popularity              0
Energy                  0
Key                     0
Loudness                0
Mode                    0
Liveness                0
Instrumentalness        0
Acousticness            0
Speechiness             0
Time Signature          0
Tempo                   0
Valence                 0
Listener                0
dtype: int64

In [60]:
# inspect missing values in the genres column
data[data['Genres'].isnull()]

# add a flag for missing genres
data['Missing Genres'] = data['Genres'].isnull()


def to_list(value):
    """Split a comma-separated cell into a clean list; missing becomes []."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]


# Artist Name(s) and Genres each hold several values per track, so store them
# as lists, and keep a text version for hover labels and display
for column in ["Artist Name(s)", "Genres"]:
    data[column] = data[column].apply(to_list)
    data[column + " Text"] = data[column].apply(", ".join)

data.head()

,Track URI,Track Name,Album Name,Artist Name(s),Popularity,Duration (ms),Explicit,Danceability,Energy,Key,...,Tempo,Time Signature,Genres,Release Date,Added At,Record Label,Listener,Missing Genres,Artist Name(s) Text,Genres Text
0,spotify:track:5SuOikwiRyPMVoIQDJUgSV,Comedy,Comedy,[Gen Hoshino],73,230666,False,0.676,0.4610,1.0,...,87.917,4.0,"[acoustic, j-pop, singer-songwriter, songwriter]",<NA>,<NA>,<NA>,Expansion,False,Gen Hoshino,"acoustic, j-pop, singer-songwriter, songwriter"
1,spotify:track:4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ghost (Acoustic),[Ben Woodward],55,149610,False,0.420,0.1660,1.0,...,77.489,4.0,"[acoustic, chill]",<NA>,<NA>,<NA>,Expansion,False,Ben Woodward,"acoustic, chill"
2,spotify:track:1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,To Begin Again,"[Ingrid Michaelson, ZAYN]",57,210826,False,0.438,0.3590,0.0,...,76.332,4.0,[acoustic],<NA>,<NA>,<NA>,Expansion,False,"Ingrid Michaelson, ZAYN",acoustic
3,spotify:track:6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Crazy Rich Asians (Original Motion Picture Sou...,[Kina Grannis],71,201933,False,0.266,0.0596,0.0,...,181.740,3.0,[acoustic],<NA>,<NA>,<NA>,Expansion,False,Kina Grannis,acoustic
4,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Hold On,[Chord Overstreet],82,198853,False,0.618,0.4430,2.0,...,119.949,4.0,[acoustic],<NA>,<NA>,<NA>,Expansion,False,Chord Overstreet,acoustic


In [61]:
# turn listener into a boolean feature per listener

# A song both of us liked used to appear once per listener, so every song
# level count, mean and correlation below counted it twice. Collapse those
# copies into one row per track and record who liked it in a column each.
listener_names = sorted(data["Listener"].unique())

# True/False per listener for every track
listener_flags = (
    data.assign(Liked=True)
    .pivot_table(
        index="Track URI",
        columns="Listener",
        values="Liked",
        aggfunc="any",
        fill_value=False,
    )
    .rename(columns={name: f"Liked by {name}" for name in listener_names})
    .reset_index()
)

flag_columns = [f"Liked by {name}" for name in listener_names]

# the song level columns are identical across copies, so keep the first one
data = (
    data.drop(columns="Listener")
    .drop_duplicates(subset="Track URI")
    .merge(listener_flags, on="Track URI")
)

# handy summaries of the flags: how many of us liked the song, and who
data["Listener Count"] = data[flag_columns].sum(axis=1)
data["Listeners"] = data[flag_columns].apply(
    lambda row: ", ".join(
        name for name, liked in zip(listener_names, row) if liked
    ),
    axis=1,
)

# Some comparisons below are per listener rather than per song. For those,
# expand back out to one row per track-listener pair; a shared song shows up
# in both listeners' panels, which is what those charts want.
by_listener = pd.concat(
    [
        data[data[f"Liked by {name}"]].assign(Listener=name)
        for name in listener_names
    ],
    ignore_index=True,
)

print(f"{len(data)} unique tracks, {len(by_listener)} track-listener pairs")
print(data["Listeners"].value_counts())
data.head()

92318 unique tracks, 92937 track-listener pairs
Listeners
Expansion                   89161
Amadou                       1491
Logan                        1068
Expansion, Logan              427
Amadou, Expansion             132
Amadou, Expansion, Logan       21
Amadou, Logan                  18
Name: count, dtype: int64


,Track URI,Track Name,Album Name,Artist Name(s),Popularity,Duration (ms),Explicit,Danceability,Energy,Key,...,Added At,Record Label,Missing Genres,Artist Name(s) Text,Genres Text,Liked by Amadou,Liked by Expansion,Liked by Logan,Listener Count,Listeners
0,spotify:track:5SuOikwiRyPMVoIQDJUgSV,Comedy,Comedy,[Gen Hoshino],73,230666,False,0.676,0.4610,1.0,...,<NA>,<NA>,False,Gen Hoshino,"acoustic, j-pop, singer-songwriter, songwriter",False,True,False,1,Expansion
1,spotify:track:4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ghost (Acoustic),[Ben Woodward],55,149610,False,0.420,0.1660,1.0,...,<NA>,<NA>,False,Ben Woodward,"acoustic, chill",False,True,False,1,Expansion
2,spotify:track:1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,To Begin Again,"[Ingrid Michaelson, ZAYN]",57,210826,False,0.438,0.3590,0.0,...,<NA>,<NA>,False,"Ingrid Michaelson, ZAYN",acoustic,False,True,False,1,Expansion
3,spotify:track:6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Crazy Rich Asians (Original Motion Picture Sou...,[Kina Grannis],71,201933,False,0.266,0.0596,0.0,...,<NA>,<NA>,False,Kina Grannis,acoustic,False,True,False,1,Expansion
4,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Hold On,[Chord Overstreet],82,198853,False,0.618,0.4430,2.0,...,<NA>,<NA>,False,Chord Overstreet,acoustic,False,True,False,1,Expansion


In [62]:
by_listener.groupby("Listener").size()

Listener
Amadou        1662
Expansion    89741
Logan         1534
dtype: int64

In [63]:
data.columns.tolist()

['Track URI',
 'Track Name',
 'Album Name',
 'Artist Name(s)',
 'Popularity',
 'Duration (ms)',
 'Explicit',
 'Danceability',
 'Energy',
 'Key',
 'Loudness',
 'Mode',
 'Speechiness',
 'Acousticness',
 'Instrumentalness',
 'Liveness',
 'Valence',
 'Tempo',
 'Time Signature',
 'Genres',
 'Release Date',
 'Added At',
 'Record Label',
 'Missing Genres',
 'Artist Name(s) Text',
 'Genres Text',
 'Liked by Amadou',
 'Liked by Expansion',
 'Liked by Logan',
 'Listener Count',
 'Listeners']

In [64]:
data.shape

(92318, 31)

In [65]:
by_listener.groupby("Listener")["Missing Genres"].sum()

Listener
Amadou       513
Expansion      0
Logan        311
Name: Missing Genres, dtype: int64

In [66]:
(
    by_listener.explode("Genres")
    .groupby("Listener")["Genres"]
    .value_counts(dropna=False)
    .groupby(level=0)
    .head(20)
)

Listener   Genres                    
Amadou     NaN                            513
           rap                            212
           r&b                            189
           melodic rap                    184
           rage rap                        78
           hip hop                         48
           alternative r&b                 43
           neo soul                        40
           cloud rap                       35
           indie                           33
           trap soul                       33
           trap                            33
           bedroom pop                     32
           jazz rap                        32
           dancehall                       28
           afrobeats                       28
           soul                            27
           afrobeat                        26
           dark r&b                        25
           drill                           23
Expansion  acoustic                      1

In [67]:
features = [
    "Valence",
    "Energy",
    "Danceability",
    "Acousticness",
    "Tempo",
    "Loudness",
    "Popularity"
]

by_listener.groupby("Listener")[features].describe()

Valence                                                       \
             count      mean       std      min      25%     50%    75%   
Listener                                                                  
Amadou      1662.0  0.486524  0.235287  0.00001  0.30200  0.4820  0.672   
Expansion  89741.0  0.469477  0.262864  0.00000  0.24900  0.4570  0.682   
Logan       1534.0  0.453042  0.215723  0.00000  0.28925  0.4245  0.600   

                   Energy            ... Loudness        Popularity  \
             max    count      mean  ...      75%    max      count   
Listener                             ...                              
Amadou     0.980   1662.0  0.600629  ...  -5.6505  0.832     1662.0   
Expansion  0.995  89741.0  0.634458  ...  -5.1080  4.532    89741.0   
Logan      0.969   1534.0  0.706815  ...  -4.4155 -1.190     1534.0   

                                                               
                mean        std  min   25%   50%   75%    max  
Listener                                                       
Amadou     57.802647  25.103174  0.0  50.0  65.0  75.0   98.0  
Expansion  33.198438  20.580824  0.0  19.0  33.0  49.0  100.0  
Logan      53.326597  23.639985  0.0  41.0  58.0  71.0   91.0  

[3 rows x 56 columns]

In [68]:
data.head()

,Track URI,Track Name,Album Name,Artist Name(s),Popularity,Duration (ms),Explicit,Danceability,Energy,Key,...,Added At,Record Label,Missing Genres,Artist Name(s) Text,Genres Text,Liked by Amadou,Liked by Expansion,Liked by Logan,Listener Count,Listeners
0,spotify:track:5SuOikwiRyPMVoIQDJUgSV,Comedy,Comedy,[Gen Hoshino],73,230666,False,0.676,0.4610,1.0,...,<NA>,<NA>,False,Gen Hoshino,"acoustic, j-pop, singer-songwriter, songwriter",False,True,False,1,Expansion
1,spotify:track:4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ghost (Acoustic),[Ben Woodward],55,149610,False,0.420,0.1660,1.0,...,<NA>,<NA>,False,Ben Woodward,"acoustic, chill",False,True,False,1,Expansion
2,spotify:track:1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,To Begin Again,"[Ingrid Michaelson, ZAYN]",57,210826,False,0.438,0.3590,0.0,...,<NA>,<NA>,False,"Ingrid Michaelson, ZAYN",acoustic,False,True,False,1,Expansion
3,spotify:track:6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Crazy Rich Asians (Original Motion Picture Sou...,[Kina Grannis],71,201933,False,0.266,0.0596,0.0,...,<NA>,<NA>,False,Kina Grannis,acoustic,False,True,False,1,Expansion
4,spotify:track:5vjLSffimiIP26QG5WcN2K,Hold On,Hold On,[Chord Overstreet],82,198853,False,0.618,0.4430,2.0,...,<NA>,<NA>,False,Chord Overstreet,acoustic,False,True,False,1,Expansion


In [69]:
fig = px.scatter(
    by_listener,
    x="Valence",
    y="Energy",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Valence vs Energy of Songs"
)

fig.show()

The graph shows that songs with higher valence often also have higher energy, but there is still a lot of variation. This suggests that valence and energy are somewhat related, but they are measuring different characteristics of a song.

In [70]:
by_listener.groupby("Listener")[features].corr()

Valence    Energy  Danceability  Acousticness  \
Listener                                                                 
Amadou    Valence       1.000000  0.364486      0.310226     -0.061257   
          Energy        0.364486  1.000000      0.003631     -0.487086   
          Danceability  0.310226  0.003631      1.000000     -0.159198   
          Acousticness -0.061257 -0.487086     -0.159198      1.000000   
          Tempo         0.031882  0.096159     -0.055989     -0.116550   
          Loudness      0.230462  0.713111      0.127263     -0.448447   
          Popularity    0.036258  0.035602     -0.081542     -0.002375   
Expansion Valence       1.000000  0.256313      0.492571     -0.103767   
          Energy        0.256313  1.000000      0.143914     -0.732566   
          Danceability  0.492571  0.143914      1.000000     -0.177043   
          Acousticness -0.103767 -0.732566     -0.177043      1.000000   
          Tempo         0.090622  0.258628     -0.021331     -0.220244   
          Loudness      0.289091  0.758774      0.271839     -0.582663   
          Popularity   -0.011526  0.013728      0.064281     -0.038847   
Logan     Valence       1.000000  0.277481      0.487273     -0.194580   
          Energy        0.277481  1.000000     -0.059519     -0.802787   
          Danceability  0.487273 -0.059519      1.000000      0.011280   
          Acousticness -0.194580 -0.802787      0.011280      1.000000   
          Tempo         0.100968  0.231618     -0.168505     -0.200262   
          Loudness      0.135323  0.737167     -0.022224     -0.601906   
          Popularity    0.128180  0.029084      0.194012     -0.069825   

                           Tempo  Loudness  Popularity  
Listener                                                
Amadou    Valence       0.031882  0.230462    0.036258  
          Energy        0.096159  0.713111    0.035602  
          Danceability -0.055989  0.127263   -0.081542  
          Acousticness -0.116550 -0.448447   -0.002375  
          Tempo         1.000000  0.059823   -0.065215  
          Loudness      0.059823  1.000000    0.060013  
          Popularity   -0.065215  0.060013    1.000000  
Expansion Valence       0.090622  0.289091   -0.011526  
          Energy        0.258628  0.758774    0.013728  
          Danceability -0.021331  0.271839    0.064281  
          Acousticness -0.220244 -0.582663   -0.038847  
          Tempo         1.000000  0.228659    0.007270  
          Loudness      0.228659  1.000000    0.071676  
          Popularity    0.007270  0.071676    1.000000  
Logan     Valence       0.100968  0.135323    0.128180  
          Energy        0.231618  0.737167    0.029084  
          Danceability -0.168505 -0.022224    0.194012  
          Acousticness -0.200262 -0.601906   -0.069825  
          Tempo         1.000000  0.186474   -0.038972  
          Loudness      0.186474  1.000000   -0.016004  
          Popularity   -0.038972 -0.016004    1.000000

The results show that energy and loudness have the strongest relationship, so songs with more energy also tend to be louder. Energy also decreases as acousticness increases. Valence and energy have a positive but moderate relationship, which supports what we saw in the scatterplot. Popularity does not seem to have a strong relationship with the other audio features.

In [71]:
fig = px.scatter(
    by_listener,
    x="Energy",
    y="Loudness",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Energy vs Loudness of Songs"
)

fig.show()

I made this graph because the correlation results showed that energy and loudness had the strongest relationship in our dataset. The graph makes that relationship easier to see: as songs become more energetic, they generally also become louder. This gives us another useful pair of features to consider when comparing songs.

In [72]:
# One row per track-genre pair, dropping tracks with no genres
genre_df = by_listener[~by_listener["Missing Genres"]].explode("Genres").copy()
genre_df = genre_df.rename(columns={"Genres": "Genre"})

# Top 5 genres for each listener separately
top_genres = (
    genre_df.groupby("Listener")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)
top_genres

Listener   Genre            
Amadou     rap                   212
           r&b                   189
           melodic rap           184
           rage rap               78
           hip hop                48
Expansion  acoustic             1000
           singer-songwriter    1000
           songwriter           1000
           indie-pop            1000
           rock                 1000
Logan      post-grunge           165
           punk                  157
           grunge                117
           hard rock             104
           metal                 103
Name: count, dtype: int64

In [73]:
# Keep only each listener's own top 5 genres
top_pairs = set(top_genres.index)

genre_summary = (
    genre_df[
        [
            pair in top_pairs
            for pair in zip(genre_df["Listener"], genre_df["Genre"])
        ]
    ]
    .groupby(["Listener", "Genre"])[["Energy", "Valence"]]
    .mean()
    .reset_index()
)

genre_summary

,Listener,Genre,Energy,Valence
0,Amadou,hip hop,0.714292,0.570885
1,Amadou,melodic rap,0.595234,0.428671
2,Amadou,r&b,0.529767,0.418516
3,Amadou,rage rap,0.631346,0.411831
4,Amadou,rap,0.592175,0.412997
5,Expansion,acoustic,0.435368,0.424023
6,Expansion,indie-pop,0.561140,0.460615
7,Expansion,rock,0.679071,0.538903
8,Expansion,singer-songwriter,0.434188,0.414298
9,Expansion,songwriter,0.434188,0.414298


In [74]:
fig = px.bar(
    genre_summary,
    x="Genre",
    y=["Energy", "Valence"],
    barmode="group",
    facet_col="Listener",
    title="Average Energy and Valence by Top Genres"
)

fig.update_xaxes(matches=None)
fig.show()

The graph shows that the top genres have different average energy and valence levels. Hip hop has the highest average energy and valence, while R&B has the lowest average energy. Rap, melodic rap, and rage rap are closer together, but still show some differences. This suggests that audio features like energy and valence may help us compare genres and could be useful later when looking for possible subgenres.

### Identifying Similarities and Differences within Genres

In [75]:
# Lets display the total distribution of genres across all listeners, not just the top 5 for each listener. Using a barplot

# data is already one row per track, so no de-duplication is needed here
unique_tracks = data

# one row per track-genre pair, dropping tracks with no genre label
all_genres = unique_tracks[~unique_tracks["Missing Genres"]].explode("Genres")
all_genres = all_genres.rename(columns={"Genres": "Genre"})

# count how many songs fall under each genre
genre_totals = all_genres["Genre"].value_counts().reset_index()
genre_totals.columns = ["Genre", "Songs"]

# there is a long tail of rare genres, so chart the 20 biggest
top_20 = genre_totals.head(20)

# genre names are long, so put them on the y axis where they have room to
# read left to right instead of being squeezed under the bars
fig = px.bar(
    top_20,
    x="Songs",
    y="Genre",
    orientation="h",
    title="Total Songs per Genre Across All Listeners",
)

# value_counts sorts biggest first, which plots bottom up, so flip the axis
# to put the biggest genre at the top
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=600)

fig.show()

## Finding: Subgenres are often nested within the set, we need to identify common pairings to see if any genres are truly JUST a subgenre.

In [76]:
# develop a measure of genre co-occurance. Loop through genre pairs and count how many times they appear together out of how many times the appear individually. If one appears together 100% of the time it appears individiually, then they are strongly associated (OR one is a true subgenre). Output should be a table, genre on the left, co-occurance rate top to bottom, and top 3-5 co occuring genres.

# reuse the exploded pairs from above, but a track can list the same genre
# twice when several of its artists share it, so keep one row per pair
genre_pairs = all_genres[["Track URI", "Genre"]].drop_duplicates()

# a genre needs enough songs before its rate means anything
MIN_TRACKS = 20
genre_sizes = genre_pairs["Genre"].value_counts()
common_genres = genre_sizes[genre_sizes >= MIN_TRACKS].index
genre_pairs = genre_pairs[genre_pairs["Genre"].isin(common_genres)]

# the set of songs under each genre, so overlap is just a set intersection
songs_by_genre = genre_pairs.groupby("Genre")["Track URI"].apply(set)

# for each genre, what share of ITS OWN songs also carry each other genre
rows = []
for genre, songs in songs_by_genre.items():
    for other_genre, other_songs in songs_by_genre.items():
        if other_genre == genre:
            continue

        shared = len(songs & other_songs)
        if shared == 0:
            continue

        rows.append({
            "Genre": genre,
            "Co-occurring Genre": other_genre,
            "Genre Songs": len(songs),
            "Shared Songs": shared,
            # 1.00 means every song in Genre also carries the other tag,
            # which is what a true subgenre looks like
            "Co-occurrence Rate": shared / len(songs),
        })

co_occurrence = pd.DataFrame(rows)

# the rate is directional: rap -> melodic rap is not the same as the
# reverse, and that gap is what tells child apart from parent
co_occurrence = co_occurrence.sort_values(
    ["Co-occurrence Rate", "Genre"], ascending=[False, True]
)

# keep each genre's 5 strongest partners
top_co_occurrence = co_occurrence.groupby("Genre").head(5)

top_co_occurrence.round(2)

,Genre,Co-occurring Genre,Genre Songs,Shared Songs,Co-occurrence Rate
17,acoustic country,country,22,22,1.0
327,ccm,christian,40,40,1.0
1115,hardcore punk,punk,41,41,1.0
2227,skate punk,punk,56,56,1.0
2264,songwriter,singer-songwriter,1000,1000,1.0
...,...,...,...,...,...
1315,j-idol,metalcore,999,1,0.0
2158,show-tunes,jazz,999,1,0.0
2478,world-music,club,999,1,0.0
2479,world-music,french,999,1,0.0


In [77]:
# lets maintain a seperate subgenre column for anything we know is a true subgenre (100% co occurance) which is a list of subgenre:parent genre keys.

# a true subgenre: every single one of its songs also carries the other tag
nested = co_occurrence[co_occurrence["Co-occurrence Rate"] == 1.0].copy()

# the parent has to be the broader label, so it needs more songs than the
# child; equal sizes would just mean two names for the same set
nested["Parent Songs"] = nested["Co-occurring Genre"].map(genre_sizes)
nested = nested[nested["Parent Songs"] > nested["Genre Songs"]]

# a subgenre can sit fully inside several parents, so keep the broadest one
nested = nested.sort_values("Parent Songs", ascending=False)
broadest_parent = nested.drop_duplicates(subset="Genre")

# the lookup: subgenre -> parent genre
SUBGENRE_PARENTS = dict(
    zip(broadest_parent["Genre"], broadest_parent["Co-occurring Genre"])
)

# tag every track-genre row, leaving the parent blank for top level labels
genre_pairs["Parent Genre"] = genre_pairs["Genre"].map(SUBGENRE_PARENTS)
genre_pairs["Is Subgenre"] = genre_pairs["Parent Genre"].notna()

# a grouped label that rolls subgenres up but leaves everything else alone,
# so we can compare genres at either level later
genre_pairs["Genre Grouped"] = genre_pairs["Parent Genre"].fillna(
    genre_pairs["Genre"]
)

print(f"{len(SUBGENRE_PARENTS)} true subgenres found")
SUBGENRE_PARENTS

6 true subgenres found


{'skate punk': 'punk',
 'hardcore punk': 'punk',
 'acoustic country': 'country',
 'songwriter': 'singer-songwriter',
 'ccm': 'christian',
 'worship': 'christian'}

In [78]:
# For each measurable, I want to see every track color coded by the genre. multicode if a track is in multiple genres. ignore subgenres.

# drop the subgenre rows so a track is not plotted twice under both rap and
# melodic rap; a track still shows up once per genuinely different genre
from IPython.display import HTML, display

top_level = genre_pairs[~genre_pairs["Is Subgenre"]][["Track URI", "Genre"]]

# attach the audio features to each track-genre pair
track_features = unique_tracks[["Track URI", "Track Name"] + features]
genre_tracks = top_level.merge(track_features, on="Track URI")

# the features live in separate columns, but faceting needs them stacked
# into one value column with the feature name alongside
long_features = genre_tracks.melt(
    id_vars=["Track URI", "Track Name", "Genre"],
    value_vars=features,
    var_name="Feature",
    value_name="Value",
)

# squeeze the facets two per row instead of all seven side by side, so each
# panel is wide enough to read
fig = px.violin(
    long_features,
    x="Value",
    y="Genre",
    color="Genre",
    facet_col="Feature",
    facet_col_wrap=2,
    points=False,
    orientation="h",
    hover_data=["Track Name"],
    title="Audio Feature Distributions by Genre",
)

# tempo, loudness and popularity are on completely different scales than
# the 0-1 features, so each panel needs its own x axis
fig.update_xaxes(matches=None, showticklabels=True, title=None)

# the facet labels already name the feature, so drop the "Feature=" prefix
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# the genre names repeat down every panel, so label them once on the left
fig.update_yaxes(title=None)

# every genre needs its own band of vertical space or the violins flatten
# into each other, so size the figure off the genre count instead of
# forcing it into a fixed height
n_genres = long_features["Genre"].nunique()
n_rows = -(-len(features) // 2)
fig.update_layout(
    height=n_rows * max(300, n_genres * 28),
    showlegend=False,
)

# that figure is far taller than the screen, so put it in a fixed-height
# box of its own that scrolls instead of swamping the notebook

display(HTML(
    '<div style="height: 700px; overflow-y: scroll;">'
    + fig.to_html(include_plotlyjs="cdn", full_html=False)
    + "</div>"
))